# Phase 5 — ML Comparative Evaluation & Model Selection: XGBoost vs. Random Forest

**Status:** ✅ Completed (pipeline v1.0)  
**Authoritative Script:** `ml_service/src/train_xgb.py`  
**Saved Models:** `models/xgb_baseline_category_v1.pkl` & `models/rf_baseline_category_v1.pkl`  
**Decision Artifact:** `artifacts/model_selection_decision_v1.md`  

---

## Executive Summary & Research Scope
Phase 5 trained the **XGBoost comparative model** using histogram-binned gradient boosting (`tree_method='hist'`, `max_depth=6`, `lr=0.1`, `n_estimators=200`, `subsample=0.8`, `colsample_bytree=0.8`) on the identical 80/20 train/test split. Both classifiers were evaluated on the full un-sampled test set (**504,160 flows**).

### Headline Empirical Comparison:
- **Multi-Class Macro F1:** XGBoost **`0.86129`** vs. Random Forest **`0.63921`** ($\Delta = +0.22208$, a massive boost in rare attack discernment).
- **False Positive Rate (FPR):** XGBoost **`0.0952%`** (399 false alarms) vs. Random Forest **`1.7665%`** (7,402 false alarms) — an **18.5× reduction in false positives**.
- **Inference Latency:** XGBoost **`14.71 μs/sample`** vs. Random Forest **`38.26 μs/sample`** (**2.6× faster inference**).
- **Binary Intrusion Detection Rate:** XGBoost **`99.802%`** vs. Random Forest **`99.940%`**.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from config.paths import Paths

with open(Paths.ARTIFACTS_DIR / 'rf_metrics_v1.json', 'r', encoding='utf-8') as f:
    rf_m = json.load(f)
with open(Paths.ARTIFACTS_DIR / 'xgb_metrics_v1.json', 'r', encoding='utf-8') as f:
    xgb_m = json.load(f)

print("Loaded empirical metrics for both Random Forest (Phase 4) and XGBoost (Phase 5).")
print(f"Test Set Sample Size: {xgb_m['test_sample_count']:,} flows")

## 1. Side-by-Side Benchmark Comparison Table
Direct metric-by-metric comparison across binary detection, multi-class categorization, and operational latency:

In [ ]:
rf_bin = rf_m['binary_intrusion_detection']
xg_bin = xgb_m['binary_intrusion_detection']
rf_glob = rf_m['global_multiclass']
xg_glob = xgb_m['global_multiclass']

bench_rows = [
    {'Metric': 'Binary Accuracy', 'Random Forest': f"{rf_bin['accuracy']*100:.3f}%", 'XGBoost': f"{xg_bin['accuracy']*100:.3f}%", 'Superior': 'XGBoost'},
    {'Metric': 'Detection Rate (TPR / Recall)', 'Random Forest': f"{rf_bin['recall_tpr']*100:.3f}%", 'XGBoost': f"{xg_bin['recall_tpr']*100:.3f}%", 'Superior': 'Random Forest'},
    {'Metric': 'False Positive Rate (FPR)', 'Random Forest': f"{rf_bin['false_positive_rate_fpr']*100:.4f}%", 'XGBoost': f"{xg_bin['false_positive_rate_fpr']*100:.4f}%", 'Superior': 'XGBoost (18.5x lower)'},
    {'Metric': 'Binary F1-Score', 'Random Forest': f"{rf_bin['f1_score']:.5f}", 'XGBoost': f"{xg_bin['f1_score']:.5f}", 'Superior': 'XGBoost'},
    {'Metric': 'Binary ROC-AUC', 'Random Forest': f"{rf_bin['roc_auc']:.5f}", 'XGBoost': f"{xg_bin['roc_auc']:.5f}", 'Superior': 'XGBoost'},
    {'Metric': 'Multiclass Accuracy (15-class)', 'Random Forest': f"{rf_glob['accuracy']*100:.3f}%", 'XGBoost': f"{xg_glob['accuracy']*100:.3f}%", 'Superior': 'XGBoost'},
    {'Metric': 'Macro F1-Score', 'Random Forest': f"{rf_glob['macro_f1']:.5f}", 'XGBoost': f"{xg_glob['macro_f1']:.5f}", 'Superior': 'XGBoost (+0.222)'},
    {'Metric': 'Weighted F1-Score', 'Random Forest': f"{rf_glob['weighted_f1']:.5f}", 'XGBoost': f"{xg_glob['weighted_f1']:.5f}", 'Superior': 'XGBoost'},
    {'Metric': 'Inference Latency', 'Random Forest': f"{rf_m['inference_latency_us_per_sample']:.2f} μs", 'XGBoost': f"{xgb_m['inference_latency_us_per_sample']:.2f} μs", 'Superior': 'XGBoost (2.6x faster)'}
]

pd.DataFrame(bench_rows)

## 2. Per-Class F1-Score Comparison Across All 15 Threat Categories
Comparing classification quality for each attack type:

In [ ]:
classes = xgb_m['classes']
comp_rows = []
for c in classes:
    rf_f1 = rf_m['per_class'][c]['f1_score']
    xg_f1 = xgb_m['per_class'][c]['f1_score']
    supp = xgb_m['per_class'][c]['support']
    comp_rows.append({
        'Category': c,
        'Support': supp,
        'RF F1': rf_f1,
        'XGBoost F1': xg_f1,
        'Delta (XGB - RF)': xg_f1 - rf_f1
    })

df_comp = pd.DataFrame(comp_rows)

# Visual comparison bar plot
df_melt = pd.melt(df_comp, id_vars=['Category'], value_vars=['RF F1', 'XGBoost F1'], var_name='Model', value_name='F1-Score')
plt.figure(figsize=(12, 6))
sns.set_theme(style='whitegrid')
sns.barplot(data=df_melt, y='Category', x='F1-Score', hue='Model', palette=['#3498db', '#e67e22'])
plt.title('Per-Class F1-Score Comparison: Random Forest vs. XGBoost', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('F1-Score', fontsize=11)
plt.ylabel('Attack Category', fontsize=11)
plt.xlim(0, 1.05)
plt.tight_layout()
plt.show()

## 3. XGBoost 15×15 Confusion Matrix Heatmap

In [ ]:
xg_cm = np.array(xgb_m['confusion_matrix'])
plt.figure(figsize=(12, 10))
sns.heatmap(np.log1p(xg_cm), annot=xg_cm, fmt='d', cmap='YlGnBu', cbar=True,
            xticklabels=classes, yticklabels=classes, annot_kws={'size': 8})
plt.title('XGBoost — 15×15 Multiclass Confusion Matrix (Color=Log1p, Text=Raw Counts)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Predicted Category', fontsize=11)
plt.ylabel('Actual Ground Truth Category', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Top 20 Feature Importance Attributions (XGBoost)

In [ ]:
xgb_booster = joblib.load(Paths.model('xgb_baseline_category_v1.pkl'))
score_dict = xgb_booster.get_score(importance_type='gain')

with open(Paths.SPLITS_DIR / 'split_summary_v1.json', 'r', encoding='utf-8') as f:
    split_meta = json.load(f)
feature_names = split_meta['features']['scaled_feature_names']

# Map f0..f75 to feature names
mapped_scores = []
for k, v in score_dict.items():
    f_idx = int(k.replace('f', ''))
    mapped_scores.append({'Feature': feature_names[f_idx], 'Gain': v})

df_xg_fi = pd.DataFrame(mapped_scores).sort_values('Gain', ascending=False).head(20)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_xg_fi, y='Feature', x='Gain', hue='Feature', palette='magma', legend=False)
plt.title('XGBoost Top 20 Features (Gain)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Average Gain Feature Importance', fontsize=11)
plt.tight_layout()
plt.show()

## 5. Formal Model Selection Verdict for Phase 6

> ### **Decision:** `XGBoost` is selected as the Primary Classifier
>
> **Rationale:**
> 1. **Massive False Positive Reduction:** FPR dropped from `1.767%` down to `0.095%` (a 94.6% reduction in SOC false alerts).
> 2. **Superior Rare Class Discernment:** Macro F1 elevated from `0.6392` to `0.8613`, with dramatic improvements on Botnet (`0.768` vs `0.264`), SSH-Patator (`0.998` vs `0.822`), and Web Attacks (`0.794` vs `0.400`).
> 3. **Lower Operational Latency:** `14.71 μs/sample` vs `38.26 μs/sample` enables higher flow throughput in live decision support.